In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# =====================================================================
# 1. 時系列ダミーデータの作成
# =====================================================================
print("--- 1. 時系列ダミーデータの作成 ---")
np.random.seed(42)

# 2026年1月1日〜2026年3月31日までの日次データを作成（約90日分）
dates = pd.date_range(start="2026-01-01", end="2026-03-31", freq="D")
n_samples = len(dates)

# トレンド + 曜日周期（7日サイクル） + ランダムノイズ で売上データをシミュレート
trend = np.linspace(100, 150, n_samples)
weekly_cycle = np.sin(2 * np.pi * dates.dayofweek / 7) * 20
noise = np.random.normal(0, 5, n_samples)
sales = trend + weekly_cycle + noise

df = pd.DataFrame({"sales": sales}, index=dates)
print(f"データ件数: {len(df)}件 (期間: {df.index.min().date()} 〜 {df.index.max().date()})")

--- 1. 時系列ダミーデータの作成 ---
データ件数: 90件 (期間: 2026-01-01 〜 2026-03-31)


In [9]:
# =====================================================================
# 2. 特徴量エンジニアリング（時系列特有の特徴量）
# =====================================================================
print("\n--- 2. 特徴量エンジニアリング（ラグ & 移動平均 & カレンダー） ---")

target_col='sales'
df['lag_1']=df[target_col].shift(1)
df['lag_7']=df[target_col].shift(7)

df['rolling_mean_3']=df[target_col].shift(1).rolling(window=3).mean()

df['dayofweek']=df.index.dayofweek
df['day']=df.index.day

df=df.dropna()

feature_cols=['lag_1','lag_7','rolling_mean_3','dayofweek','day']
X=df[feature_cols]
y=df[target_col]

split_date=df.index[-15]
split_date




--- 2. 特徴量エンジニアリング（ラグ & 移動平均 & カレンダー） ---


Timestamp('2026-03-17 00:00:00')